# Rolling Window Evaluation Results

Anchored rolling-window backtest: 8 windows, train anchored at 2000, ~2y test each.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Resolve project root: allow running from src/visualizations/ or project root
ROOT = Path.cwd()
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

with open(ROOT / 'artifacts/backtest/walkforward_metrics.json') as f:
    metrics = json.load(f)

with open(ROOT / 'artifacts/models/rolling_eval_results.json') as f:
    rolling = json.load(f)

wf_df = pd.read_parquet(ROOT / 'artifacts/backtest/walkforward_results.parquet')

## Overall Performance (All Windows Combined)

In [10]:
# Summary table
summary = pd.DataFrame(metrics).T[['cumulative_return', 'annualized_return', 'sharpe', 'sortino', 'max_drawdown', 'win_rate']]
summary = summary.round(4)
summary['cumulative_return'] = summary['cumulative_return'].apply(lambda x: f"{x:.2f}x")
summary['annualized_return'] = summary['annualized_return'].apply(lambda x: f"{x:.1%}")
summary['max_drawdown'] = summary['max_drawdown'].apply(lambda x: f"{x:.1%}")
summary['win_rate'] = summary['win_rate'].apply(lambda x: f"{x:.1%}")
display(summary)

,cumulative_return,annualized_return,sharpe,sortino,max_drawdown,win_rate
agent,39.89x,26.6%,1.1158,1.3745,47.9%,54.9%
equal_weight,29.55x,24.3%,1.1171,1.3231,48.4%,55.3%
market_cap,7.27x,14.4%,0.6347,0.8180,42.4%,52.7%
inv_vol,10.66x,16.9%,0.8715,0.9889,46.1%,54.9%


## Per-Window Metrics Distribution

In [11]:
# Extract per-window stats
summary_stats = rolling['summary']

metrics_list = ['cumulative_return', 'sharpe', 'max_drawdown', 'annualized_return']
strategies = ['agent', 'equal_weight', 'inv_vol']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=metrics_list,
    specs=[[{'type': 'box'}, {'type': 'box'}],
           [{'type': 'box'}, {'type': 'box'}]]
)

colors = {'agent': '#636EFA', 'equal_weight': '#EF553B', 'inv_vol': '#00CC96'}
positions = {'agent': 1, 'equal_weight': 2, 'inv_vol': 3}

for metric_idx, metric in enumerate(metrics_list, 1):
    row = (metric_idx - 1) // 2 + 1
    col = (metric_idx - 1) % 2 + 1
    
    for strat in strategies:
        data = summary_stats[strat][metric]
        values = [data['min'], data['min'] + (data['max'] - data['min']) * 0.25,
                 data['min'] + (data['max'] - data['min']) * 0.5,
                 data['min'] + (data['max'] - data['min']) * 0.75, data['max']]
        
        fig.add_trace(
            go.Box(
                y=[data['min'], data['max']],
                name=strat,
                marker_color=colors[strat],
                boxmean='sd',
                showlegend=(metric_idx == 1),
                legendgroup=strat
            ),
            row=row, col=col
        )

fig.update_layout(height=700, title_text="Per-Window Metric Distributions", showlegend=True)
fig.show()

## Cumulative Value Trajectory

In [12]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=wf_df['date'], y=wf_df['value_agent'], mode='lines', name='Agent', line=dict(color='#636EFA')))
fig.add_trace(go.Scatter(x=wf_df['date'], y=wf_df['value_equal_weight'], mode='lines', name='Equal-Weight', line=dict(color='#EF553B')))
fig.add_trace(go.Scatter(x=wf_df['date'], y=wf_df['value_inv_vol'], mode='lines', name='Inv-Vol', line=dict(color='#00CC96')))
fig.add_trace(go.Scatter(x=wf_df['date'], y=wf_df['value_market_cap'], mode='lines', name='Market-Cap', line=dict(color='#AB63FA')))

fig.update_layout(
    title="Portfolio Value Over Time (Anchored Rolling Windows)",
    xaxis_title="Date",
    yaxis_title="Portfolio Value",
    hovermode='x unified',
    height=500
)
fig.update_yaxes(type='log')
fig.show()

## Agent vs Market Benchmark (BOVA11 / IBOV)

In [13]:
# Buy-and-hold BOVA11 (IBOV proxy), normalized to the agent's starting value on the same dates
dates = pd.to_datetime(wf_df['date'])
start_val = wf_df['value_agent'].iloc[0]

bova = (pd.read_parquet(ROOT / 'data/raw/br/prices/BOVA11.parquet')
        .set_index('trade_date')['adj_close'].sort_index())
bova = bova.reindex(dates).ffill().bfill()
bova_curve = bova.to_numpy() / bova.iloc[0] * start_val

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=wf_df['value_agent'], mode='lines', name='Agent', line=dict(color='#636EFA', width=3)))
fig.add_trace(go.Scatter(x=dates, y=wf_df['value_equal_weight'], mode='lines', name='Equal-Weight', line=dict(color='#EF553B', width=1.5)))
fig.add_trace(go.Scatter(x=dates, y=bova_curve, mode='lines', name='BOVA11 (IBOV)', line=dict(color='#FFA15A', width=2, dash='dash')))

fig.update_layout(
    title="Agent vs Market Benchmark (BOVA11 buy-and-hold, log scale)",
    xaxis_title="Date", yaxis_title="Portfolio Value",
    hovermode='x unified', height=500
)
fig.update_yaxes(type='log')
fig.show()

# Benchmark metrics over the same out-of-sample window
import sys
sys.path.insert(0, str(ROOT))
from src.agent.metrics import compute_all

bova_rets = np.log(pd.Series(bova_curve) / pd.Series(bova_curve).shift(1)).fillna(0.0).to_numpy()
bova_metrics = compute_all(bova_rets, bova_curve)
print("BOVA11 (IBOV proxy) over full OOS window:")
print(f"  Cumulative return: {bova_metrics['cumulative_return']:.2f}x")
print(f"  Annualized return: {bova_metrics['annualized_return']:.1%}")
print(f"  Sharpe:            {bova_metrics['sharpe']:.3f}")
print(f"  Max drawdown:      {bova_metrics['max_drawdown']:.1%}")
print(f"\nAgent excess Sharpe vs BOVA11: {metrics['agent']['sharpe'] - bova_metrics['sharpe']:+.3f}")

BOVA11 (IBOV proxy) over full OOS window:
  Cumulative return: 1.29x
  Annualized return: 5.4%
  Sharpe:            0.222
  Max drawdown:      49.7%

Agent excess Sharpe vs BOVA11: +0.894


## Agent vs Fixed Income (SELIC / CDI), Inflation (IPCA) & Big Tickers

In [14]:
# All benchmarks normalized to the agent's starting value on the agent's dates.
# Reuses `dates`, `start_val`, `compute_all` from the BOVA11 cell above.
agent_curve = wf_df['value_agent'].to_numpy()

def price_curve(ticker):
    """Buy-and-hold value of a stock, normalized to start_val on the agent's dates."""
    px = (pd.read_parquet(ROOT / f'data/raw/br/prices/{ticker}.parquet')
          .set_index('trade_date')['adj_close'].sort_index())
    px = px.reindex(dates).ffill().bfill()
    return px.to_numpy() / px.iloc[0] * start_val

def macro_series(name):
    s = pd.read_parquet(ROOT / f'data/raw/br/macro/{name}.parquet')
    return s.set_index('reference_date')[name].sort_index()

def rate_curve(name):
    """Compound a daily-rate series (SELIC/CDI, percent per day) into a value curve."""
    r = macro_series(name).reindex(dates).ffill().bfill().to_numpy() / 100.0
    return np.cumprod(1 + r) * start_val

def ipca_curve():
    """Compound monthly IPCA (%) into a cumulative-inflation value curve."""
    ipca = macro_series('ipca')
    ipca = ipca[ipca.index >= dates.min() - pd.Timedelta(days=40)]
    cum = (1 + ipca / 100).cumprod().reindex(dates, method='ffill')
    return (cum / cum.iloc[0] * start_val).to_numpy()

curves = {
    'Agent': agent_curve,
    'SELIC': rate_curve('selic'),
    'CDI': rate_curve('cdi'),
    'IPCA (inflation)': ipca_curve(),
    'PETR4': price_curve('PETR4'),
    'ITSA4': price_curve('ITSA4'),   # ITSA3 not in universe; ITSA4 is the liquid Itausa share
    'WEGE3': price_curve('WEGE3'),
}

styles = {'Agent': dict(color='#636EFA', width=3),
          'SELIC': dict(color='#00CC96', width=2, dash='dash'),
          'CDI': dict(color='#19D3F3', width=2, dash='dash'),
          'IPCA (inflation)': dict(color='#EF553B', width=2, dash='dot')}

fig = go.Figure()
for name, curve in curves.items():
    fig.add_trace(go.Scatter(x=dates, y=curve, mode='lines', name=name,
                             line=styles.get(name, dict(width=1.6))))
fig.update_layout(title="Agent vs SELIC / CDI / IPCA & Big Tickers (R$ start, log scale)",
                  xaxis_title="Date", yaxis_title="Value (R$)",
                  hovermode='x unified', height=550)
fig.update_yaxes(type='log')
fig.show()

# Metrics table over the same OOS window
rows = {}
for name, curve in curves.items():
    rets = np.log(pd.Series(curve) / pd.Series(curve).shift(1)).fillna(0.0).to_numpy()
    rows[name] = compute_all(rets, curve)
bench = pd.DataFrame(rows).T[['cumulative_return', 'annualized_return', 'sharpe', 'max_drawdown']]
fmt = bench.copy()
for c in ['cumulative_return', 'annualized_return', 'max_drawdown']:
    fmt[c] = fmt[c].map('{:+.1%}'.format)
fmt['sharpe'] = bench['sharpe'].round(3)
display(fmt)

,cumulative_return,annualized_return,sharpe,max_drawdown
Agent,+3945.6%,+26.5%,1.113,+47.9%
SELIC,+333.6%,+9.8%,45.186,+0.0%
CDI,+331.8%,+9.7%,45.036,+0.0%
IPCA (inflation),+144.3%,+5.8%,2.816,+1.3%
PETR4,+1280.3%,+18.2%,0.369,+85.2%
ITSA4,+1220.7%,+17.8%,0.556,+43.8%
WEGE3,+1773.1%,+20.5%,0.594,+49.8%


## Daily Returns Distribution

In [15]:
fig = go.Figure()

fig.add_trace(go.Histogram(x=wf_df['log_return'], name='Daily Returns', nbinsx=50, marker_color='#636EFA'))

fig.update_layout(
    title="Daily Log Return Distribution",
    xaxis_title="Log Return",
    yaxis_title="Frequency",
    height=400
)
fig.show()

print(f"Return Stats:")
print(f"  Mean: {wf_df['log_return'].mean():.4f}")
print(f"  Std:  {wf_df['log_return'].std():.4f}")
print(f"  Min:  {wf_df['log_return'].min():.4f}")
print(f"  Max:  {wf_df['log_return'].max():.4f}")

Return Stats:
  Mean: 0.0009
  Std:  0.0133
  Min:  -0.1480
  Max:  0.1057


## Window Performance Breakdown

In [16]:
# Per-window breakdown (metrics nested under w['metrics'])
windows = rolling['windows']
window_df = pd.DataFrame([
    {
        'window': w['window_id'],
        'test_period': f"{w['test_start']} → {w['test_end']}",
        'agent_sharpe': w['metrics']['agent']['sharpe'],
        'agent_return': w['metrics']['agent']['cumulative_return'],
        'equal_weight_sharpe': w['metrics']['equal_weight']['sharpe'],
        'inv_vol_sharpe': w['metrics']['inv_vol']['sharpe'],
        'market_cap_sharpe': w['metrics']['market_cap']['sharpe'],
    }
    for w in windows
])

x = window_df['test_period']
fig = go.Figure()
fig.add_trace(go.Bar(x=x, y=window_df['agent_sharpe'], name='Agent', marker_color='#636EFA'))
fig.add_trace(go.Bar(x=x, y=window_df['equal_weight_sharpe'], name='Equal-Weight', marker_color='#EF553B'))
fig.add_trace(go.Bar(x=x, y=window_df['inv_vol_sharpe'], name='Inv-Vol', marker_color='#00CC96'))
fig.add_trace(go.Bar(x=x, y=window_df['market_cap_sharpe'], name='Market-Cap', marker_color='#AB63FA'))

fig.update_layout(
    title="Sharpe Ratio per Rolling Window",
    xaxis_title="Test Period",
    yaxis_title="Sharpe Ratio",
    barmode='group',
    height=450
)
fig.show()

display(window_df.round(3))

,window,test_period,agent_sharpe,agent_return,equal_weight_sharpe,inv_vol_sharpe,market_cap_sharpe
0,0,2010-01-03 → 2012-01-03,0.896,0.599,0.934,0.727,0.713
1,1,2012-01-04 → 2014-01-04,1.421,0.411,1.307,1.203,0.358
2,2,2014-01-05 → 2016-01-05,-0.389,-0.118,-0.543,-0.670,-0.321
3,3,2016-01-06 → 2018-01-06,3.149,2.185,3.349,3.088,1.990
4,4,2018-01-07 → 2020-01-07,3.073,1.500,3.073,2.784,1.372
5,5,2020-01-08 → 2022-01-08,0.310,0.224,0.268,0.067,0.009
6,6,2022-01-09 → 2024-01-09,1.059,0.575,1.136,0.452,1.033
7,7,2024-01-10 → 2026-01-10,0.883,0.338,0.843,0.800,0.569


## Agent vs Baselines Comparison

In [17]:
# Create comparison metrics
comparison = {
    'Agent': metrics['agent'],
    'Equal-Weight': metrics['equal_weight'],
    'Inv-Vol': metrics['inv_vol'],
    'Market-Cap': metrics['market_cap']
}

comp_df = pd.DataFrame(comparison).T
comp_df = comp_df[['sharpe', 'sortino', 'annualized_return', 'max_drawdown', 'win_rate']]
comp_df = comp_df.round(4)

# Highlight agent row
def highlight_agent(row):
    return ['background-color: lightblue' if 'Agent' in str(row.name) else '' for _ in row]

display(comp_df.style.apply(highlight_agent, axis=1))

,sharpe,sortino,annualized_return,max_drawdown,win_rate
Agent,1.115800,1.374500,0.266000,0.478600,0.549300
Equal-Weight,1.117100,1.323100,0.242800,0.484000,0.553300
Inv-Vol,0.871500,0.988900,0.168900,0.461000,0.548500
Market-Cap,0.634700,0.818000,0.143700,0.424400,0.527100


## Model Stock Composition

What the agent actually holds: top holdings over time, portfolio concentration, and sector exposure.

In [18]:
weight_cols = [c for c in wf_df.columns if c.startswith('w_')]
W = wf_df[weight_cols].to_numpy()

# --- Concentration: top-10 share + effective number of stocks (inverse HHI) ---
top10_share = np.sort(W, axis=1)[:, -10:].sum(axis=1)
n_effective = 1.0 / (W ** 2).sum(axis=1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=wf_df['date'], y=top10_share * 100, name='top-10 weight share (%)'))
fig.add_trace(go.Scatter(x=wf_df['date'], y=n_effective, name='effective # stocks', yaxis='y2'))
fig.update_layout(title='Concentration: Top-10 Share & Effective Diversification',
                  yaxis_title='Top-10 share (%)',
                  yaxis2=dict(title='Effective # stocks', overlaying='y', side='right'),
                  hovermode='x unified', height=450)
fig.show()

# --- Top-10 holdings over time (by mean weight), stacked ---
mean_w = wf_df[weight_cols].mean().sort_values(ascending=False).head(10)
fig = go.Figure()
for col in mean_w.index:
    fig.add_trace(go.Scatter(x=wf_df['date'], y=wf_df[col] * 100,
                             name=col.removeprefix('w_'), stackgroup='w'))
fig.update_layout(title='Top 10 Holdings Over Time (stacked)', yaxis_title='Weight (%)',
                  hovermode='x unified', height=450)
fig.show()

print("Top 10 holdings by average weight:")
display((mean_w * 100).round(2).rename('avg weight (%)').rename_axis('ticker')
        .rename(lambda t: t.removeprefix('w_')))

Top 10 holdings by average weight:


ticker
TELB4    2.31
COCE5    1.06
MYPK3    0.96
CCTY3    0.93
LEVE3    0.86
BRFS3    0.79
ALPA4    0.78
MILS3    0.78
LPSB3    0.77
CPFE3    0.77
Name: avg weight (%), dtype: float64

In [19]:
# --- Sector exposure over time (top 8 sectors by mean weight) ---
sector_path = ROOT / 'data/processed/ml_dataset_training.parquet'
if sector_path.exists():
    sectors = (pd.read_parquet(sector_path, columns=['ticker', 'sector'])
               .drop_duplicates('ticker').set_index('ticker')['sector'])
    col_sector = {c: sectors.get(c.removeprefix('w_'), 'Unknown') for c in weight_cols}
    sector_w = wf_df[weight_cols].T.groupby(pd.Series(col_sector)).sum().T
    sector_w.index = wf_df['date']
    top_sectors = sector_w.mean().sort_values(ascending=False).head(8).index

    fig = go.Figure()
    for s in top_sectors:
        fig.add_trace(go.Scatter(x=sector_w.index, y=sector_w[s] * 100, name=str(s)[:35], stackgroup='s'))
    fig.update_layout(title='Sector Exposure (top 8 sectors)', yaxis_title='Weight (%)',
                      hovermode='x unified', height=480)
    fig.show()

    print("Average sector allocation (%):")
    display((sector_w.mean() * 100).sort_values(ascending=False).round(2).rename('avg weight (%)'))
else:
    print(f"Sector data not found at {sector_path} — skipping sector breakdown.")

Average sector allocation (%):


Construção Civil, Mat. Constr. e Decoração                 9.98
Comércio (Atacado e Varejo)                                7.83
Energia Elétrica                                           7.08
Máquinas, Equipamentos, Veículos e Peças                   7.07
Metalurgia e Siderurgia                                    6.54
Bancos                                                     6.21
Têxtil e Vestuário                                         5.88
Serviços Transporte e Logística                            5.30
Emp. Adm. Part. - Const. Civil, Mat. Const. e Decoração    3.82
Telecomunicações                                           3.57
Serviços Médicos                                           2.76
Agricultura (Açúcar, Álcool e Cana)                        2.70
Comunicação e Informática                                  2.52
Alimentos                                                  2.23
Petroquímicos e Borracha                                   2.01
Farmacêutico e Higiene                  